In [2]:
!python -m pip install --upgrade pip
!python -m pip install --upgrade pandas openpyxl anthropic ipykernel

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 4.1 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.3 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.7 MB 3.5 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/10.7 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.4/10.7 MB 4.0 MB/s eta 0:00:03
   ----------- ---------------------------- 3.1/10.7 MB 3.8 MB/s eta 0:00:02
   -------------- ------------------------- 3.9/10.7 MB

This cell checks the new documents which were submitted and their current status signature.

In [3]:
import pandas as pd

# Read the document status file, keeping every column as a string
doc_status_df = pd.read_excel("DocumentStatus.xlsx", dtype=str)

# Keep only the documents whose Status is null
null_status_df = doc_status_df.loc[
    doc_status_df["Status"].isna(), ["KYCAssessmentID", "DocumentNo", "CreationDate", "LastUpdated"]
].reset_index(drop=True)

print(f"Documents with null status: {len(null_status_df)} of {len(doc_status_df)}")
null_status_df

Documents with null status: 3 of 3


,KYCAssessmentID,DocumentNo,CreationDate,LastUpdated
0,921632,001,2026-11-10 00:00:00,NaN
1,927015,002,2026-11-10 00:00:00,NaN
2,954350,003,2026-11-10 00:00:00,NaN


This cell loads the PDF files which do not have a Status yet.

In [4]:
import base64
from pathlib import Path

DOCUMENTS_DIR = Path("Documents")

def read_pdf(document_no):
    """Return the raw bytes of Documents/<DocumentNo>.pdf, or None if it is missing."""
    pdf_path = DOCUMENTS_DIR / f"{document_no}.pdf"
    return pdf_path.read_bytes() if pdf_path.exists() else None

# Read the PDF for each document with a null status
null_status_df["PdfBytes"] = null_status_df["DocumentNo"].apply(read_pdf)
null_status_df["PdfBase64"] = null_status_df["PdfBytes"].apply(
    lambda b: base64.standard_b64encode(b).decode("utf-8") if b is not None else None
)

missing = null_status_df.loc[null_status_df["PdfBytes"].isna(), "DocumentNo"].tolist()
print(f"PDFs loaded: {len(null_status_df) - len(missing)} of {len(null_status_df)}")
if missing:
    print(f"Missing PDFs for DocumentNo: {missing}")
null_status_df[["KYCAssessmentID", "DocumentNo"]].assign(
    PdfSizeKB=null_status_df["PdfBytes"].apply(lambda b: round(len(b) / 1024, 1) if b else None)
)

PDFs loaded: 3 of 3


,KYCAssessmentID,DocumentNo,PdfSizeKB
0,921632,001,1117.9
1,927015,002,861.7
2,954350,003,195.1


This cell checks for digital signatures. If a digital signature is detected, then it updates the status to "Digital Signature Detected", and changes the LastUpdated status to the time of the runtime.

In [5]:
import re
from datetime import datetime

# A signed PDF contains a signature dictionary with a /ByteRange entry that lists the
# signed byte ranges. Empty (unsigned) signature fields have no /ByteRange.
BYTE_RANGE_PATTERN = re.compile(rb"/ByteRange\s*\[")

def has_digital_signature(pdf_bytes):
    """Return True if the PDF contains at least one applied digital signature."""
    return pdf_bytes is not None and BYTE_RANGE_PATTERN.search(pdf_bytes) is not None

null_status_df["HasDigitalSignature"] = null_status_df["PdfBytes"].apply(has_digital_signature)

# Timestamp for every status changed in this run, as a string to match dtype=str
last_updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Update the status of digitally signed documents
signed_mask = null_status_df["HasDigitalSignature"]
null_status_df["Status"] = None
null_status_df["LastUpdated"] = None
null_status_df.loc[signed_mask, "Status"] = "Digital Signature Detected"
null_status_df.loc[signed_mask, "LastUpdated"] = last_updated

# Carry the new status back to the full document status dataframe
signed_document_nos = null_status_df.loc[signed_mask, "DocumentNo"]
doc_status_df.loc[
    doc_status_df["DocumentNo"].isin(signed_document_nos), ["Status", "LastUpdated"]
] = ["Digital Signature Detected", last_updated]

print(f"Digital signatures detected: {signed_mask.sum()} of {len(null_status_df)}")
null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]

Digital signatures detected: 0 of 3


,KYCAssessmentID,DocumentNo,HasDigitalSignature,Status,LastUpdated
0,921632,001,False,None,None
1,927015,002,False,None,None
2,954350,003,False,None,None


This section of code checks for the remaining documents which do not have a digital signature to see if there is a manual or no signature and updates the status of the document accordingly.

In [ ]:
import os
from getpass import getpass

# --- User credentials -------------------------------------------------------
# AUTH_METHOD:
#   "api_key" - use the user's Anthropic API key
#   "profile" - use the user's `ant auth login` profile (no key needed)
AUTH_METHOD = "api_key"

# API key: taken from the ANTHROPIC_API_KEY environment variable, otherwise prompted for.
# Do not paste the key into this notebook.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if AUTH_METHOD == "api_key" and not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Anthropic API key: ").strip()

# Profile name for AUTH_METHOD = "profile"; None uses the active profile
ANTHROPIC_PROFILE = None

In [ ]:
import anthropic

def get_client(auth_method=AUTH_METHOD, api_key=ANTHROPIC_API_KEY, profile=ANTHROPIC_PROFILE):
    """Build an authenticated Anthropic client from the user's credentials.

    auth_method:
      "api_key" - authenticate with the user's API key
      "profile" - authenticate with the user's `ant auth login` profile
    """
    if auth_method == "api_key":
        if not api_key:
            raise ValueError("No API key provided. Set ANTHROPIC_API_KEY in the credentials cell.")
        return anthropic.Anthropic(api_key=api_key)
    if auth_method == "profile":
        # An API key in the environment would take precedence over the profile, so remove it
        os.environ.pop("ANTHROPIC_API_KEY", None)
        os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
        if profile:
            os.environ["ANTHROPIC_PROFILE"] = profile
        return anthropic.Anthropic()
    raise ValueError(f"Unknown AUTH_METHOD '{auth_method}'. Use 'api_key' or 'profile'.")

client = get_client()
print(f"Anthropic client ready (auth method: {AUTH_METHOD})")

In [ ]:
import json

# `client` is created in the authentication cell above
MODEL = "claude-opus-5-5"
SIGNATURE_PROMPT = "Does this file contain a signature? Return with options Manual Signature and No Signature"
SIGNATURE_OPTIONS = ["Manual Signature", "No Signature"]

def classify_signature(pdf_base64):
    """Send one PDF to Claude and return "Manual Signature" or "No Signature", or None on failure."""
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=1024,
        # On a safety refusal, the API retries the request on a fallback model
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        output_config={
            "effort": "low",
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {"signature": {"type": "string", "enum": SIGNATURE_OPTIONS}},
                    "required": ["signature"],
                    "additionalProperties": False,
                },
            },
        },
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_base64},
                },
                {"type": "text", "text": SIGNATURE_PROMPT},
            ],
        }],
    )
    if response.stop_reason != "end_turn":
        print(f"Unexpected stop reason '{response.stop_reason}' (request {response._request_id})")
        return None
    text = next(b.text for b in response.content if b.type == "text")
    return json.loads(text)["signature"]

# Send the documents that still have a null status (and a loaded PDF) one at a time
to_classify = null_status_df.index[
    null_status_df["Status"].isna() & null_status_df["PdfBase64"].notna()
]
print(f"Documents to send to Claude: {len(to_classify)}")

for idx in to_classify:
    document_no = null_status_df.at[idx, "DocumentNo"]
    try:
        signature = classify_signature(null_status_df.at[idx, "PdfBase64"])
    except anthropic.RateLimitError as e:
        print(f"{document_no}: rate limited, retry after {e.response.headers.get('retry-after')}s")
        continue
    except anthropic.APIStatusError as e:
        print(f"{document_no}: API error {e.status_code}: {e.message}")
        continue
    except anthropic.APIConnectionError:
        print(f"{document_no}: network error")
        continue
    if signature is None:
        continue

    # Update the status in both dataframes
    last_updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    null_status_df.loc[idx, ["Status", "LastUpdated"]] = [signature, last_updated]
    doc_status_df.loc[
        doc_status_df["DocumentNo"] == document_no, ["Status", "LastUpdated"]
    ] = [signature, last_updated]
    print(f"{document_no}: {signature}")

null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]